# Import libraries

In [10]:
import torch
import torch.nn as nn
from torch.utils.data import DataLoader, TensorDataset
import pandas as pd
import numpy as np
from sklearn.preprocessing import StandardScaler, MinMaxScaler
from sklearn.model_selection import train_test_split
from scipy.stats import entropy
import matplotlib.pyplot as plt
import seaborn as sns
from torch.utils.data import Sampler
import torch.optim as optim
from sklearn.decomposition import PCA
import tensorflow as tf

# Load data and labels

In [4]:
real_data = pd.read_csv("esol_dataset.csv")
real_data1 = real_data.iloc[:,:-2]
threshold = []
batch_size = 100
# ECFP fingerprints
X = real_data1.values.astype(np.float32)
# Actual solubility values
y = real_data.iloc[:, -2:-1].values.astype(np.float32)
scaler = StandardScaler()
input_dim = X.shape[1]
latent_dim = 128
label_dim = 1
epochs = 100

# Encoder

In [5]:
def build_conditional_encoder(
    input_dim: int,
    label_dim: int,
    latent_dim: int,
    hidden_layers=(512,256,128),
    activation="elu",
    use_label_embedding: bool = False,
    label_embed_dim: int = 16
):
    x_in = tf.keras.Input(shape=(input_dim,), name="x")
    y_in = tf.keras.Input(shape=(label_dim,), name="y")

    y_feat = y_in
    if use_label_embedding:
        y_feat = tf.keras.layers.Dense(
            label_embed_dim,
            activation=activation,
            name="y_embed"
        )(y_in)

    h = tf.keras.layers.Concatenate(name="enc_concat")([x_in, y_feat])

    for i, units in enumerate(hidden_layers):
        h = tf.keras.layers.Dense(
            units,
            activation=activation,
            name=f"enc_dense_{i}"
        )(h)

    z_mean = tf.keras.layers.Dense(latent_dim, name="z_mean")(h)
    z_logvar = tf.keras.layers.Dense(latent_dim, name="z_logvar")(h)

    return tf.keras.Model([x_in, y_in], [z_mean, z_logvar], name="cvae_encoder")

# Decoder

In [6]:
def build_conditional_decoder(
    input_dim: int,
    label_dim: int,
    latent_dim: int,
    hidden_layers=(128, 256, 512),
    activation="gelu",
    output_activation=None,
    use_label_embedding: bool = False,
    label_embed_dim: int = 16,
):
    z_in = tf.keras.Input(shape=(latent_dim,), name="z")
    y_in = tf.keras.Input(shape=(label_dim,), name="y")

    y_feat = y_in
    if use_label_embedding:
        y_feat = tf.keras.layers.Dense(
            label_embed_dim,
            activation=activation,
            name="y_embed"
        )(y_in)

    h = tf.keras.layers.Concatenate(name="dec_concat")([z_in, y_feat])

    for i, units in enumerate(hidden_layers):
        h = tf.keras.layers.Dense(
            units,
            activation=activation,
            name=f"dec_dense_{i}"
        )(h)

    x_logits = tf.keras.layers.Dense(
        input_dim,
        activation=None,
        name="x_logits"
    )(h)

    return tf.keras.Model([z_in, y_in], x_logits, name="cvae_decoder")


# Binary sampling

The decoder outputs logits. During generation, sigmoid converts them to Bernoulli probabilities and each ECFP bit is sampled as exactly 0 or 1.


In [7]:
def sampling(z_mean, z_logvar):
    """
    Reparameterization trick.
    """
    eps = tf.random.normal(tf.shape(z_mean))
    return z_mean + tf.exp(0.5 * z_logvar) * eps

# VAE Model

In [8]:
class ConditionalVAE(tf.keras.Model):
    def __init__(
        self,
        encoder: tf.keras.Model,
        decoder: tf.keras.Model,
        beta: float = 0.1,
        max_beta: float = None,
        warmup_epochs: int = 0,
        recon_loss: str = "bce",
        name="ConditionalVAE",
    ):
        super().__init__(name=name)
        self.encoder = encoder
        self.decoder = decoder

        self.beta = float(beta)
        self.max_beta = float(max_beta) if max_beta is not None else None
        self.warmup_epochs = int(warmup_epochs)
        self.recon_loss = recon_loss.lower()

        self.loss_tracker = tf.keras.metrics.Mean(name="loss")
        self.recon_tracker = tf.keras.metrics.Mean(name="recon")
        self.kl_tracker = tf.keras.metrics.Mean(name="kl")
        self.beta_tracker = tf.keras.metrics.Mean(name="beta")
        self._epoch_var = tf.Variable(0, dtype=tf.int32, trainable=False)

    @property
    def metrics(self):
        return [
            self.loss_tracker,
            self.recon_tracker,
            self.kl_tracker,
            self.beta_tracker,
        ]

    def set_epoch(self, epoch: int):
        self._epoch_var.assign(int(epoch))

    def _current_beta(self):
        if self.warmup_epochs <= 0 or self.max_beta is None:
            return tf.constant(self.beta, dtype=tf.float32)

        e = tf.cast(self._epoch_var, tf.float32)
        w = tf.cast(self.warmup_epochs, tf.float32)
        frac = tf.clip_by_value(e / w, 0.0, 1.0)

        return (
            tf.constant(self.beta, tf.float32)
            + frac * (
                tf.constant(self.max_beta, tf.float32)
                - tf.constant(self.beta, tf.float32)
            )
        )

    def _reconstruction_loss(self, x, x_logits):
        # ECFP fingerprints are binary (0/1), so use Bernoulli likelihood.
        # from_logits=True is more numerically stable than applying sigmoid
        # before BCE.
        per_dim = tf.nn.sigmoid_cross_entropy_with_logits(
            labels=x,
            logits=x_logits
        )

        # Sum over fingerprint bits, then average over the batch.
        return tf.reduce_mean(tf.reduce_sum(per_dim, axis=1))

    def train_step(self, data):
        (x, y), _ = data

        with tf.GradientTape() as tape:
            z_mean, z_logvar = self.encoder([x, y], training=True)
            z = sampling(z_mean, z_logvar)

            # Decoder returns logits.
            x_logits = self.decoder([z, y], training=True)

            recon = self._reconstruction_loss(x, x_logits)

            kl = tf.reduce_mean(
                -0.5 * tf.reduce_sum(
                    1 + z_logvar
                    - tf.square(z_mean)
                    - tf.exp(z_logvar),
                    axis=1,
                )
            )

            beta = self._current_beta()
            loss = recon + beta * kl

        grads = tape.gradient(loss, self.trainable_variables)
        self.optimizer.apply_gradients(
            zip(grads, self.trainable_variables)
        )

        self.loss_tracker.update_state(loss)
        self.recon_tracker.update_state(recon)
        self.kl_tracker.update_state(kl)
        self.beta_tracker.update_state(beta)

        return {m.name: m.result() for m in self.metrics}

    def call(self, inputs, training=False):
        x, y = inputs

        z_mean, z_logvar = self.encoder(
            [x, y],
            training=training
        )

        z = sampling(z_mean, z_logvar)

        # Return logits, consistent with the decoder.
        return self.decoder([z, y], training=training)


class EpochSetter(tf.keras.callbacks.Callback):
    def on_epoch_begin(self, epoch, logs=None):
        if hasattr(self.model, "set_epoch"):
            self.model.set_epoch(epoch)


# Train the model

In [9]:
encoder = build_conditional_encoder(
    input_dim=input_dim,
    label_dim=label_dim,
    latent_dim=latent_dim,
    hidden_layers=[512,256,128],
    activation="elu",
    use_label_embedding=True,
)

decoder = build_conditional_decoder(
    input_dim=input_dim,
    label_dim=label_dim,
    latent_dim=latent_dim,
    hidden_layers=[128,256,512],
    activation="gelu",
    output_activation=None,
    use_label_embedding=True,
)

model = ConditionalVAE(
    encoder=encoder,
    decoder=decoder,
    beta=0.1,
    max_beta=1,
    warmup_epochs=80,
    recon_loss="bce",
)

model.compile(
    optimizer=tf.keras.optimizers.Adam(5e-4)
)

# -----------------------------
# Train
# -----------------------------
history = model.fit(
    x=(X,y),
    y=X,
    epochs=epochs,
    batch_size=batch_size,
    shuffle=True,
    callbacks=[EpochSetter()]
)


2026-10-06 13:41:11.046651: W tensorflow/stream_executor/platform/default/dso_loader.cc:64] Could not load dynamic library 'libcuda.so.1'; dlerror: libcuda.so.1: cannot open shared object file: No such file or directory
2026-10-06 13:41:11.048161: W tensorflow/stream_executor/cuda/cuda_driver.cc:263] failed call to cuInit: UNKNOWN ERROR (303)
2026-10-06 13:41:11.048286: I tensorflow/stream_executor/cuda/cuda_diagnostics.cc:156] kernel driver does not appear to be running on this host (DESKTOP-LFUOV61): /proc/driver/nvidia/version does not exist
2026-10-06 13:41:11.051084: I tensorflow/core/platform/cpu_feature_guard.cc:193] This TensorFlow binary is optimized with oneAPI Deep Neural Network Library (oneDNN) to use the following CPU instructions in performance-critical operations:  AVX2 AVX512F AVX512_VNNI FMA
To enable them in other operations, rebuild TensorFlow with the appropriate compiler flags.


Epoch 1/100
12/12 [==============================] - 1s 16ms/step - loss: 561.5516 - recon: 559.8351 - kl: 17.1649 - beta: 0.1000
Epoch 2/100
12/12 [==============================] - 0s 20ms/step - loss: 167.4561 - recon: 148.5295 - kl: 170.1269 - beta: 0.1113
Epoch 3/100
12/12 [==============================] - 0s 14ms/step - loss: 106.6404 - recon: 101.2808 - kl: 43.7516 - beta: 0.1225
Epoch 4/100
12/12 [==============================] - 0s 13ms/step - loss: 92.0710 - recon: 87.3924 - kl: 34.9804 - beta: 0.1337
Epoch 5/100
12/12 [==============================] - 0s 13ms/step - loss: 86.3217 - recon: 83.1386 - kl: 21.9519 - beta: 0.1450
Epoch 6/100
12/12 [==============================] - 0s 14ms/step - loss: 84.8421 - recon: 81.8634 - kl: 19.0634 - beta: 0.1562
Epoch 7/100
12/12 [==============================] - 0s 16ms/step - loss: 83.1541 - recon: 80.6142 - kl: 15.1633 - beta: 0.1675
Epoch 8/100
12/12 [==============================] - 0s 15ms/step - loss: 82.4787 - recon: 80.123

# Binary sampling

The decoder outputs logits. During generation, sigmoid converts them to Bernoulli probabilities and each ECFP bit is sampled as exactly 0 or 1.


In [14]:
target_solubility = np.percentile(y,90)
column_names = real_data1.columns
# Convert the desired solubility to the same standardized
# scale used during training
y_target = np.array([[target_solubility]], dtype=np.float32).astype(np.float32).reshape(-1)

def generate_conditioned(decoder, y_cond, n=1000, temperature=1.0):

    y_cond = tf.convert_to_tensor(
        y_cond,
        dtype=tf.float32
    )

    if len(y_cond.shape) == 1:
        y_cond = tf.repeat(
            y_cond[None, :],
            repeats=n,
            axis=0
        )

    z = tf.random.normal(
        (tf.shape(y_cond)[0], latent_dim)
    )

    # Decoder returns logits
    logits = decoder(
        [z, y_cond],
        training=False
    )

    # Convert logits to Bernoulli probabilities
    probs = tf.sigmoid(logits / temperature)

    # Sample binary ECFP bits
    x_gen = tf.cast(
        tf.random.uniform(tf.shape(probs)) < probs,
        tf.float32
    )

    return x_gen


x_gen = generate_conditioned(
    decoder,
    y_target,
    n=1000,
    temperature=1
)

def postprocess_generated(
    x_gen,
    scaler,
    feature_names,
    log_type="ln_eps",
    eps=1e-8
):
    # tensor -> numpy
    if hasattr(x_gen, "numpy"):
        x_gen = x_gen.numpy()

    # Ensure binary values.
    x_gen = np.rint(x_gen).astype(np.int8)

    return pd.DataFrame(
        x_gen,
        columns=feature_names
    )


df_gen = postprocess_generated(
    x_gen,
    scaler=scaler,
    feature_names=column_names,
    log_type="ln_eps"
)




In [16]:
df_gen.to_csv('gen_ESOL_class1_cont_90.csv')